# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mishellscripts/flyrank/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The method I choose to use is the "yes/no" with observed label using logistic regression. This label attempts to a answer a question of: Did a declining page recover by the next time period? This fits the refresh/content opportunity scoring because it studies the refresh potential of a declining page. I will use logistic regression to start, with the coefficients providing explanations to important features, use PCA if necessary to reduce correlation, and introduce random forest/boost methods for precision.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Training can only occur on client data with enough history. A cutoff date T will be selected and tuned to train the model using future (after T) recovery data and past (before T) decline data. Clients are to be split into train/test.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [29]:
T = "2025-06-01"  # set from the coverage check above

# --- feature set, as of T only, no label-derived fields ---
content_type_query = f"""
    SELECT content_hash_id, content_type, DATEDIFF('day', DATE '{T}', content_updated_date) AS days_since_last_update
    FROM read_parquet('{rel}/dim_content.parquet')
"""
content_dim = con.sql(content_type_query).df()

feature_query = f"""
    SELECT client_hash_id, content_hash_id,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date <= DATE '{T}'
"""
X_raw = con.sql(feature_query).df().merge(content_dim, on="content_hash_id", how="left")

# --- gate: is_declining at T ---
gate_query = f"""
    WITH windowed AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN DATE '2025-05-02' AND DATE '{T}'
                     THEN gsc_impressions ELSE 0 END) AS impr_last30,
            SUM(CASE WHEN report_date BETWEEN DATE '2025-04-01' AND DATE '2025-05-01'
                     THEN gsc_impressions ELSE 0 END) AS impr_prev30
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date <= DATE '{T}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN (impr_last30 - impr_prev30) / NULLIF(impr_prev30, 0) * 100 < -20
             THEN 1 ELSE 0 END AS is_declining_at_T
    FROM windowed
"""
gate_df = con.sql(gate_query).df()

# --- recovery label: strictly after T ---
future_query = f"""
    WITH future AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN DATE '2025-07-01' AND DATE '2025-07-31'
                     THEN gsc_impressions ELSE 0 END) AS impr_T1,
            SUM(CASE WHEN report_date BETWEEN DATE '2025-06-02' AND DATE '2025-07-01'
                     THEN gsc_impressions ELSE 0 END) AS impr_baseline
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date > DATE '{T}' AND report_date <= DATE '2025-07-31'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN impr_T1 > impr_baseline THEN 1 ELSE 0 END AS recovered_by_T1
    FROM future
"""
recovery_df = con.sql(future_query).df()

In [40]:
print(X_raw.isna().sum())

client_hash_id            0
content_hash_id           0
gsc_avg_position          3
gsc_impressions           0
gsc_clicks                0
content_type              0
days_since_last_update    0
dtype: int64


In [41]:
X_raw["gsc_avg_position"] = X_raw["gsc_avg_position"].fillna(0)

In [42]:
print(X_raw.isna().sum())

client_hash_id            0
content_hash_id           0
gsc_avg_position          0
gsc_impressions           0
gsc_clicks                0
content_type              0
days_since_last_update    0
dtype: int64


In [43]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

declining_ids = gate_df.loc[gate_df["is_declining_at_T"] == 1, ["client_hash_id", "content_hash_id"]]

data = (X_raw.merge(declining_ids, on=["client_hash_id", "content_hash_id"], how="inner")
             .merge(recovery_df[["client_hash_id", "content_hash_id", "recovered_by_T1"]],
                     on=["client_hash_id", "content_hash_id"], how="inner"))

y = data.pop("recovered_by_T1")
groups = data["client_hash_id"]
X = pd.get_dummies(data.drop(columns=["client_hash_id", "content_hash_id"]), columns=["content_type"])

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"train: {len(X_train)} rows, {groups.iloc[train_idx].nunique()} clients")
print(f"test:  {len(X_test)} rows, {groups.iloc[test_idx].nunique()} clients")

train: 106990 rows, 3 clients
test:  22816 rows, 1 clients


In [44]:
import numpy as np

logit = LogisticRegression(random_state=42, class_weight="balanced", max_iter=1000).fit(X_train, y_train)
rf = RandomForestClassifier(max_depth=4, random_state=42, class_weight="balanced").fit(X_train, y_train)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

stayed_broken = 1 - y_test.values

results = []
for name, scores in [
    ("logistic regression (1 - P(recovery))", 1 - logit.predict_proba(X_test)[:, 1]),
    ("random forest (1 - P(recovery))", 1 - rf.predict_proba(X_test)[:, 1]),
]:
    for k in (20, 50, 200):
        results.append((name, k, precision_at_k(scores, stayed_broken, k)))

comparison = pd.DataFrame(results, columns=["method", "k", "precision"]).pivot(index="method", columns="k", values="precision")
comparison["base_rate"] = stayed_broken.mean()
print(comparison)

k                                        20    50    200  base_rate
method                                                             
logistic regression (1 - P(recovery))  0.20  0.30  0.355   0.193811
random forest (1 - P(recovery))        0.45  0.26  0.230   0.193811


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [45]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(5))

predictions = rf.predict(X_test)
wrong_mask = predictions != y_test.values
wrong = X_test[wrong_mask].copy()
wrong["predicted_recovery"] = predictions[wrong_mask]
wrong["actual_recovery"] = y_test.values[wrong_mask]
print(wrong.head(3))

days_since_last_update          0.546216
gsc_impressions                 0.221426
gsc_avg_position                0.212637
gsc_clicks                      0.019721
content_type_keyword article    0.000000
dtype: float64
   gsc_avg_position  gsc_impressions  gsc_clicks  days_since_last_update  \
1          71.60000                5           0                     353   
2           6.65625               32           0                     351   
3           8.00000                1           0                     351   

   content_type_keyword article  predicted_recovery  actual_recovery  
1                          True                   1                0  
2                          True                   0                1  
3                          True                   0                1  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.